# Benchmarking Local vs Hub Algorithms

This tutorial shows how to **benchmark local algorithms against community-contributed hub algorithms** using TuiML's experiment framework:

1. **Browse** the TuiML Hub for community algorithms
2. **Install** a hub algorithm
3. **Benchmark** it against a local (built-in) algorithm
4. **Compare** results with statistical tests and export formats

## 1. Setup

In [1]:
import tuiml
import tuiml.algorithms  # Register all built-in algorithms
from tuiml.hub.remote import remote

# Quick check: how many local algorithms are available?
classifiers = tuiml.list_algorithms(type="classifier")
print(f"Local classifiers available: {len(classifiers)}")
for algo in classifiers[:5]:
    print(f"  - {algo['name']}")
print(f"  ... and {len(classifiers) - 5} more")

Local classifiers available: 42
  - NaiveBayesClassifier
  - NaiveBayesMultinomialClassifier
  - BayesianNetworkClassifier
  - DecisionStumpClassifier
  - C45TreeClassifier
  ... and 37 more


## 2. Browse the Hub for Community Algorithms

The TuiML Hub hosts community-contributed algorithms. Let's see what classifiers are available.

In [2]:
# Browse hub for all algorithms
hub_algos = remote.browse()
print(f"Hub algorithms: {len(hub_algos)}")
for algo in hub_algos:
    print(f"  - {algo.name}  (type={algo.algorithm_type}, category={algo.category})")

# Filter for classifiers
hub_classifiers = [a for a in hub_algos if a.algorithm_type == "classifier"]
print(f"\nHub classifiers: {len(hub_classifiers)}")
for algo in hub_classifiers:
    print(f"  - {algo.name}")

Hub algorithms: 7
  - simple_decision_tree  (type=classifier, category=tree)
  - MRMRSelector  (type=feature_selection, category=filter)
  - QuantileTransformer  (type=preprocessor, category=scaling)
  - TargetEncoder  (type=preprocessor, category=encoding)
  - SAINTClassifier  (type=classifier, category=deep_learning)
  - NODEClassifier  (type=classifier, category=deep_learning)
  - TabNetClassifier  (type=classifier, category=deep_learning)

Hub classifiers: 4
  - simple_decision_tree
  - SAINTClassifier
  - NODEClassifier
  - TabNetClassifier


## 3. Install a Hub Algorithm

Install `SAINTClassifier` from the hub - a deep learning classifier contributed by the community.

In [3]:
# Install and load a hub classifier
remote.install("SAINTClassifier")
HubClassifier = remote.load("SAINTClassifier")

# Verify it has the expected interface
hub_model = HubClassifier()
print(f"Loaded: {HubClassifier.__name__}")
print(f"  has fit():    {hasattr(hub_model, 'fit')}")
print(f"  has predict(): {hasattr(hub_model, 'predict')}")

Algorithm 'SAINTClassifier' is already installed. Use force=True to reinstall.
Loaded: SAINTClassifier
  has fit():    True
  has predict(): True


## 4. Benchmark: Local vs Hub

Compare the local `RandomForestClassifier` against the hub's `SAINTClassifier` using 5-fold cross-validation on the iris dataset.

In [4]:
# Benchmark: 1 local algorithm vs 1 hub algorithm
from tuiml.algorithms.trees import RandomForestClassifier

exp = tuiml.experiment(
    algorithms={
        "RandomForest (local)": RandomForestClassifier(),
        "SAINTClassifier (hub)": HubClassifier(),
    },
    datasets=["iris"],
    cv=5,
)

print(exp.summary())

Experiment: Experiment
Validation: cross_validation
Metric: accuracy

Dataset: iris
--------------------------------------------------
  RandomForest (local): 0.9467 ± 0.0452
  SAINTClassifier (hub): 0.3267 ± 0.0133



## 5. Results & Export

In [5]:
# Markdown table
print(exp.to_markdown())

| Dataset | RandomForest (local) | SAINTClassifier (hub) |
|---|---|---|
| iris | 0.9467 ± 0.0506 | 0.3267 ± 0.0149 ▼ |
|---|---|---|
| **W/L/T** | 0/0/1 | 0/1/0 |


## 6. Statistical Comparison

In [6]:
# Pairwise comparison: hub vs local
comparisons = exp.compare_models(
    baseline="RandomForest (local)",
    metric="accuracy",
)

for key, stats in comparisons.items():
    print(f"{stats['dataset']}:")
    print(f"  {stats['model']}: {stats['model_mean']:.4f}")
    print(f"  {stats['baseline']}: {stats['baseline_mean']:.4f}")
    print(f"  p-value: {stats['p_value']:.4f}  significant: {'Yes' if stats['significant'] else 'No'}")
    print()

iris:
  SAINTClassifier (hub): 0.3267
  RandomForest (local): 0.9467
  p-value: 0.0000  significant: Yes

